In [ ]:
import pandas as pd
from threadpoolctl import threadpool_limits

from study.data import cohort, features, prediction_metadata
from study.io import WORK, load, reset_directory, save, write_table
from study.metrics import probability_metrics
from study.models import apply_calibration, fit_source_model, predict
from study.statistics import prediction_rows

In [ ]:
parameters = {
    'n_estimators': 250,
    'max_depth': 3,
    'learning_rate': 0.04,
    'min_child_weight': 5,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'reg_lambda': 3,
    'objective': 'binary:logistic',
    'eval_metric': 'logloss',
    'importance_type': 'gain',
    'random_state': 77,
    'n_jobs': 1
}

CV_FOLDS = 5
CV_SEED = 77
ALLOCATION_REPEATS = tuple(range(20))

transitions = [
    ('carevue', 'carevue'),
    ('carevue', 'metavision'),
    ('metavision', 'metavision'),
    ('metavision', 'mimiciv'),
    ('metavision', 'eicu')
]

In [ ]:
# Check source and evaluation populations
feature_sets = load(WORK / 'feature_sets.pkl')
fit_ids = {}
checks = []

for domain, role in [('carevue', 'source'), ('metavision', 'source'), ('mimiciv', 'test'), ('eicu', 'test')]:
    population = cohort(domain).query('partition == @role').sort_index()
    assert population.index.is_unique and population.subject_id.is_unique
    
    hospitals = population.hospital_id.nunique() if domain == 'eicu' else 1

    # Check reserved calibration and test populations are disjoint
    if domain == 'eicu':
        reserved = cohort(domain).query("partition == 'calibration'")
        assert not set(population.subject_id) & set(reserved.subject_id)

    # Retain all eligible source patients for fitting
    if domain in ('carevue', 'metavision'):
        fit_ids[domain, 0] = population.index

    checks.append(dict(domain = domain, role = role, patients = len(population), hospitals = hospitals))

write_table(WORK / 'results/cohort_checks.parquet', pd.DataFrame(checks))

In [ ]:
# Check CareVue and MetaVision patients are disjoint
carevue, metavision = cohort('carevue'), cohort('metavision')
assert not set(carevue.subject_id) & set(metavision.subject_id)

In [ ]:
# Fit source models
models, cv_rows = {}, []

for source in ('carevue', 'metavision'):
    metadata, x = cohort(source).sort_index(), features(source)
    ids = fit_ids[source, 0]
    supports = ('original',) if source == 'carevue' else ('original', 'shared')

    # Fit original and shared feature tiers
    for support in supports:
        for tier, columns in feature_sets[support].items():
            with threadpool_limits(limits = 1):
                fitted = fit_source_model(
                    x.loc[ids, columns], metadata.loc[ids, 'in_hospital_mortality'],
                    'xgboost', parameters, folds = CV_FOLDS, seed = CV_SEED,
                )
            models[source, 0, support, tier] = fitted

            # Summarise predictions from each held-out fold
            for fold in range(CV_FOLDS):
                held_out = fitted['oof_fold'].index[fitted['oof_fold'].eq(fold)]
                cv_rows.append(
                    dict(
                        source = source, support = support, tier = tier, fold = fold,
                        **probability_metrics(
                            metadata.loc[held_out, 'in_hospital_mortality'], fitted['oof'].loc[held_out]
                        ),
                    )
                )

    print(source, len(ids), 'source patients;', CV_FOLDS, 'folds')

save(
    WORK / 'models/source.pkl',
    {
        'models': models,
        'fit_ids': fit_ids,
        'parameters': parameters,
        'repeats': (0,),
        'allocation_repeats': ALLOCATION_REPEATS,
        'design': 'source_cv',
        'cv_folds': CV_FOLDS,
        'cv_seed': CV_SEED
    },
)
write_table(WORK / 'results/source_cv.parquet', pd.DataFrame(cv_rows))

In [ ]:
# Generate source OOF and complete target predictions
reset_directory(WORK / 'predictions/full')
reset_directory(WORK / 'predictions/full_source_calibrated')

for source, target in transitions:
    metadata, x = cohort(target).sort_index(), features(target)
    ids = metadata.index
    supports = ('shared',) if target == 'eicu' else ('original',) if source == 'carevue' else ('original', 'shared')
    rows, corrected = [], []

    # Use OOF source risks and full-fit target risks
    for support in supports:
        for tier in ('R0', 'R1', 'R2', 'R3', 'R4'):
            fitted = models[source, 0, support, tier]
            risk = fitted['oof'].loc[ids].to_numpy() if source == target else predict(fitted['model'], x.loc[ids])
            evaluated = prediction_metadata(metadata.loc[ids])
            if source == target:
                evaluated = evaluated.assign(partition = 'source_oof')

            settings = dict(
                domain = target, source = source, support = support, tier = tier, repeat = 0, restriction = 'full'
            )
            rows.append(prediction_rows(evaluated, risk, **settings))

            # Apply Platt correction to external cohorts
            if source == 'metavision' and target != source:
                corrected.append(
                    prediction_rows(evaluated, apply_calibration(fitted['corrections']['platt'], risk), **settings)
                )

    write_table(WORK / 'predictions/full' / f'{source}_{target}.parquet', pd.concat(rows, ignore_index = True))
    if corrected:
        write_table(
            WORK / 'predictions/full_source_calibrated' / f'{target}.parquet',
            pd.concat(corrected, ignore_index = True),
        )